# HashMap / HashSet — The #1 Interview Pattern

If you only master ONE data structure for interviews, make it the hash map.

**Why?** The majority of array/string interview problems have an O(n²) brute force that
can be optimized to O(n) with a hash map or hash set. Interviewers EXPECT you to reach
for this tool.

---

## What This Lesson Covers

1. Core Concept: Why Hash Maps Exist
2. Python Tools: `dict`, `set`, `defaultdict`, `Counter`
3. **The 7 HashMap/Set Patterns** (with full execution guides)
4. **How to SPOT the Pattern** in an interview
5. Complexity Analysis
6. Common Gotchas
7. Interview Execution Framework

---
## 1. Core Concept: Why Hash Maps Exist

The fundamental problem: **lookup**.

| Operation | List/Array | Hash Map/Set |
|-----------|-----------|---------------|
| Search for value | O(n) | **O(1)** |
| Check membership | O(n) | **O(1)** |
| Count occurrences | O(n) per query | **O(1)** after O(n) build |
| Find complement | O(n) inner loop → O(n²) | **O(1)** lookup → O(n) total |

**The core insight:** Whenever you have a nested loop where the inner loop is searching
for something, you can almost always replace it with a hash map lookup.

```
Brute Force O(n²)          Optimized O(n)
for i in range(n):    →    seen = {}  
    for j in range(n):     for x in arr:
        if match...            if complement in seen:
```

---
## 2. Python Tools for HashMap/Set Problems

In [ ]:
# === dict — the standard hash map ===
# Use when: you need to map keys to values

# Creating
freq = {}                         # Empty dict
freq = {"a": 1, "b": 2}          # Literal
freq = dict.fromkeys("abc", 0)   # All keys start at 0

# Key operations
freq["a"] = 1                    # Set
val = freq["a"]                  # Get (KeyError if missing)
val = freq.get("z", 0)          # Get with default (safe)
"a" in freq                      # O(1) membership check

# Counting pattern
text = "hello"
freq = {}
for ch in text:
    freq[ch] = freq.get(ch, 0) + 1
print(freq)  # {'h': 1, 'e': 1, 'l': 2, 'o': 1}

In [ ]:
# === set — the hash set ===
# Use when: you only care about membership (no values needed)

seen = set()                      # Empty set
seen = {1, 2, 3}                  # Literal

# Key operations
seen.add(4)                       # Add element — O(1)
seen.discard(2)                   # Remove if present — O(1)
4 in seen                         # Membership — O(1)

# Set operations
a = {1, 2, 3}
b = {2, 3, 4}
print(a & b)    # Intersection: {2, 3}
print(a | b)    # Union: {1, 2, 3, 4}
print(a - b)    # Difference: {1}
print(a ^ b)    # Symmetric difference: {1, 4}

In [ ]:
# === defaultdict — auto-initializing dict ===
# Use when: you're building lists/sets/counts per key

from collections import defaultdict

# Grouping pattern — no need to check if key exists
groups = defaultdict(list)
words = ["eat", "tea", "tan", "ate", "nat", "bat"]
for word in words:
    key = tuple(sorted(word))  # Anagram signature
    groups[key].append(word)

print(dict(groups))
# {('a','e','t'): ['eat','tea','ate'], ('a','n','t'): ['tan','nat'], ('a','b','t'): ['bat']}

# Counting pattern
freq = defaultdict(int)
for ch in "mississippi":
    freq[ch] += 1
print(dict(freq))  # {'m': 1, 'i': 4, 's': 4, 'p': 2}

In [ ]:
# === Counter — specialized counting dict ===
# Use when: you need to count things (it's the fastest way)

from collections import Counter

freq = Counter("mississippi")
print(freq)                    # Counter({'i': 4, 's': 4, 'p': 2, 'm': 1})
print(freq.most_common(2))     # [('i', 4), ('s', 4)]
print(freq['x'])               # 0 (no KeyError for missing keys)

# Comparing frequencies (anagram check!)
print(Counter("listen") == Counter("silent"))  # True

# Counter arithmetic
a = Counter("aab")
b = Counter("abb")
print(a - b)    # Counter({'a': 1})  — what a has that b doesn't
print(a & b)    # Counter({'a': 1, 'b': 1})  — minimum of each
print(a | b)    # Counter({'a': 2, 'b': 2})  — maximum of each

---
## 3. The 7 HashMap/Set Patterns

These are the recurring templates you'll see over and over. Learn to recognize them
and you can solve most hash-based problems.

| # | Pattern | Core Idea | Classic Problem |
|---|---------|-----------|------------------|
| 1 | Complement Lookup | Store seen values, check for complement | Two Sum |
| 2 | Frequency Map | Count occurrences, query counts | Valid Anagram |
| 3 | Seen Set / Duplicate Detection | Track what you've visited | Contains Duplicate |
| 4 | Group By Key | Bucket items by computed key | Group Anagrams |
| 5 | Index Map | Store value→index for position queries | Two Sum (return indices) |
| 6 | Prefix Sum + HashMap | Running sum + count of prior sums | Subarray Sum Equals K |
| 7 | Sliding Window + HashMap | Window frequency tracking | Longest Substring Without Repeat |

---
### Pattern 1: Complement Lookup

**The Idea:** Instead of searching for a match with a nested loop, store what you've
seen and check if the *complement* (the value you need) exists.

**When to use:** Any time you need to find **two values** that satisfy a condition
(sum, difference, product, etc.).

**Template:**
```python
def find_pair(arr, target):
    seen = {}  # or set()
    for x in arr:
        complement = target - x  # What do I NEED?
        if complement in seen:
            return (complement, x)  # Found it
        seen[x] = True  # or seen.add(x)
    return None
```

**Mental model:** Think of it as walking through a party asking:
"Have I already met the person who completes my pair?"

In [ ]:
# Pattern 1 Example: Two Sum
# Given nums and target, find two numbers that add to target.

# BRUTE FORCE — O(n²)
def two_sum_brute(nums, target):
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] + nums[j] == target:
                return [i, j]
    return []

# OPTIMIZED — O(n) with hash map
def two_sum(nums, target):
    seen = {}  # value → index
    for i, num in enumerate(nums):
        complement = target - num
        if complement in seen:
            return [seen[complement], i]
        seen[num] = i
    return []

# Test
print(two_sum([2, 7, 11, 15], 9))   # [0, 1]
print(two_sum([3, 2, 4], 6))         # [1, 2]
print(two_sum([3, 3], 6))            # [0, 1]

In [ ]:
# Pattern 1 Variant: Two Sum — return boolean (even simpler)
# Use a SET instead of dict when you don't need indices

def has_pair_with_sum(nums, target):
    seen = set()
    for num in nums:
        if target - num in seen:
            return True
        seen.add(num)
    return False

print(has_pair_with_sum([2, 7, 11, 15], 9))  # True
print(has_pair_with_sum([2, 7, 11, 15], 5))  # False

**Key insight:** The complement lookup works for ANY relationship, not just sum:
- Sum: `complement = target - num`
- Difference: `complement = num - target` or `target + num`
- Product: `complement = target / num`
- XOR: `complement = target ^ num`

---
### Pattern 2: Frequency Map

**The Idea:** Count how often each element appears. Then query those counts to
answer the question.

**When to use:**
- "Is this an anagram?"
- "Find the most/least frequent element"
- "Find elements that appear exactly N times"
- "Can you rearrange to form...?"
- Any problem involving **counting** or **frequency comparison**

**Template:**
```python
from collections import Counter

def frequency_based(arr):
    freq = Counter(arr)       # Build frequency map
    # Now query it:
    # freq[x]                 → count of x
    # freq.most_common(k)     → top k elements
    # freq == other_freq      → same distribution?
```

In [ ]:
# Pattern 2 Example: Valid Anagram
# Two strings are anagrams if they have the same character frequencies.

from collections import Counter

def is_anagram(s, t):
    return Counter(s) == Counter(t)

print(is_anagram("anagram", "nagaram"))  # True
print(is_anagram("rat", "car"))           # False

# Manual version (to understand what Counter does)
def is_anagram_manual(s, t):
    if len(s) != len(t):
        return False
    freq = {}
    for ch in s:
        freq[ch] = freq.get(ch, 0) + 1
    for ch in t:
        freq[ch] = freq.get(ch, 0) - 1
        if freq[ch] < 0:
            return False
    return True

print(is_anagram_manual("anagram", "nagaram"))  # True

In [ ]:
# Pattern 2 Example: Top K Frequent Elements
# Given nums, return the k most frequent elements.

from collections import Counter

def top_k_frequent(nums, k):
    freq = Counter(nums)
    return [x for x, _ in freq.most_common(k)]

print(top_k_frequent([1,1,1,2,2,3], 2))  # [1, 2]
print(top_k_frequent([1], 1))              # [1]

# Interview follow-up: What if you can't use most_common?
# Use bucket sort! Index = frequency, value = list of elements
def top_k_bucket(nums, k):
    freq = Counter(nums)
    # Buckets: index i holds elements that appear i times
    buckets = [[] for _ in range(len(nums) + 1)]
    for num, count in freq.items():
        buckets[count].append(num)
    
    result = []
    for i in range(len(buckets) - 1, -1, -1):  # Highest freq first
        for num in buckets[i]:
            result.append(num)
            if len(result) == k:
                return result
    return result

print(top_k_bucket([1,1,1,2,2,3], 2))  # [1, 2]

---
### Pattern 3: Seen Set / Duplicate Detection

**The Idea:** Maintain a set of values you've already processed. Use it to detect
duplicates, find first unique, or check constraints.

**When to use:**
- "Contains duplicate"
- "Find the first non-repeating element"
- "Have I visited this state before?" (BFS/DFS)
- "Is this value unique?"

**Template:**
```python
def has_duplicates(arr):
    seen = set()
    for x in arr:
        if x in seen:
            return True  # Duplicate!
        seen.add(x)
    return False
```

**One-liner variant:** `len(set(arr)) != len(arr)`

In [ ]:
# Pattern 3 Example: Contains Duplicate

def contains_duplicate(nums):
    seen = set()
    for num in nums:
        if num in seen:
            return True
        seen.add(num)
    return False

print(contains_duplicate([1, 2, 3, 1]))  # True
print(contains_duplicate([1, 2, 3, 4]))  # False

# One-liner:
def contains_duplicate_v2(nums):
    return len(set(nums)) != len(nums)

print(contains_duplicate_v2([1, 2, 3, 1]))  # True

In [ ]:
# Pattern 3 Example: Longest Consecutive Sequence
# Given unsorted array, find length of longest consecutive sequence.
# [100, 4, 200, 1, 3, 2] → 4 (sequence: 1, 2, 3, 4)

def longest_consecutive(nums):
    num_set = set(nums)  # O(1) lookups
    best = 0
    
    for num in num_set:
        # Only start counting from the BEGINNING of a sequence
        # How do we know it's the start? num-1 is NOT in the set
        if num - 1 not in num_set:
            current = num
            streak = 1
            
            while current + 1 in num_set:
                current += 1
                streak += 1
            
            best = max(best, streak)
    
    return best

print(longest_consecutive([100, 4, 200, 1, 3, 2]))  # 4
print(longest_consecutive([0, 3, 7, 2, 5, 8, 4, 6, 0, 1]))  # 9

**Key insight for Longest Consecutive:** The trick is to only start counting from sequence
beginnings (`num - 1 not in set`). This makes it O(n) even though there's a while loop
inside — each number is visited at most twice.

---
### Pattern 4: Group By Key

**The Idea:** Compute a *signature* or *key* for each element. Group elements with
the same key together using a hash map.

**When to use:**
- "Group anagrams"
- "Find all items that share property X"
- "Bucket items by some computed category"

**Template:**
```python
from collections import defaultdict

def group_by(items):
    groups = defaultdict(list)
    for item in items:
        key = compute_key(item)  # Define what groups them
        groups[key].append(item)
    return list(groups.values())
```

**The hard part** is choosing the right key function.

In [ ]:
# Pattern 4 Example: Group Anagrams
# Group words that are anagrams of each other.

from collections import defaultdict

def group_anagrams(strs):
    groups = defaultdict(list)
    for s in strs:
        # Key: sorted characters as tuple (tuples are hashable, lists aren't)
        key = tuple(sorted(s))
        groups[key].append(s)
    return list(groups.values())

print(group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]))
# [['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]

In [ ]:
# Pattern 4 Alternative key: character frequency tuple
# Faster than sorting for long strings: O(n) vs O(n log n)

from collections import defaultdict

def group_anagrams_v2(strs):
    groups = defaultdict(list)
    for s in strs:
        # Key: count of each letter (26 slots for a-z)
        counts = [0] * 26
        for ch in s:
            counts[ord(ch) - ord('a')] += 1
        groups[tuple(counts)].append(s)
    return list(groups.values())

print(group_anagrams_v2(["eat", "tea", "tan", "ate", "nat", "bat"]))

---
### Pattern 5: Index Map

**The Idea:** Store `value → index` (or `value → position`) so you can quickly
find WHERE a value is, not just IF it exists.

**When to use:**
- "Return the indices" (not just the values)
- "Find the distance between matching elements"
- "Find the last/first occurrence"

**Template:**
```python
def with_indices(arr):
    index_map = {}  # value → index
    for i, val in enumerate(arr):
        if condition(val, index_map):
            return result_using(i, index_map[val])
        index_map[val] = i
```

In [ ]:
# Pattern 5 Example: Contains Duplicate II
# Return True if there are two equal elements with indices at most k apart.

def contains_nearby_duplicate(nums, k):
    index_map = {}  # value → last seen index
    for i, num in enumerate(nums):
        if num in index_map and i - index_map[num] <= k:
            return True
        index_map[num] = i  # Update to latest index
    return False

print(contains_nearby_duplicate([1,2,3,1], 3))    # True (indices 0,3)
print(contains_nearby_duplicate([1,0,1,1], 1))    # True (indices 2,3)
print(contains_nearby_duplicate([1,2,3,1,2,3], 2))  # False

---
### Pattern 6: Prefix Sum + HashMap

**The Idea:** This is one of the most powerful patterns. Maintain a running sum
and use a hash map to count how many times each prefix sum has occurred.

**The Trick:** If `prefix_sum[j] - prefix_sum[i] == k`, then the subarray from
`i+1` to `j` sums to `k`. So for each position, ask:
"Has `current_sum - k` appeared before as a prefix sum?"

**When to use:**
- "Subarray sum equals K"
- "Count subarrays with sum K"
- "Longest subarray with sum K"
- "Subarray sum divisible by K"
- Any problem about **contiguous subarray** + **target sum**

**Template:**
```python
def subarray_sum(nums, k):
    prefix_count = {0: 1}  # Base case: empty prefix has sum 0
    current_sum = 0
    result = 0
    
    for num in nums:
        current_sum += num
        complement = current_sum - k
        if complement in prefix_count:
            result += prefix_count[complement]
        prefix_count[current_sum] = prefix_count.get(current_sum, 0) + 1
    
    return result
```

**Why `{0: 1}` initially?** Because if `current_sum == k` at any point, the subarray
from index 0 to here sums to k. We need `current_sum - k = 0` to be in the map.

In [ ]:
# Pattern 6 Example: Subarray Sum Equals K
# Count the number of contiguous subarrays that sum to k.

def subarray_sum(nums, k):
    prefix_count = {0: 1}  # sum → how many times we've seen this sum
    current_sum = 0
    count = 0
    
    for num in nums:
        current_sum += num
        # How many previous prefix sums equal current_sum - k?
        # Each one represents a subarray that sums to k
        complement = current_sum - k
        if complement in prefix_count:
            count += prefix_count[complement]
        prefix_count[current_sum] = prefix_count.get(current_sum, 0) + 1
    
    return count

print(subarray_sum([1, 1, 1], 2))        # 2 ([1,1] at idx 0-1 and 1-2)
print(subarray_sum([1, 2, 3], 3))        # 2 ([1,2] and [3])
print(subarray_sum([1, -1, 1, 1, -1], 1))  # 5

In [ ]:
# Let's trace through prefix sum step by step
# nums = [1, 2, 3], k = 3

nums = [1, 2, 3]
k = 3
prefix_count = {0: 1}
current_sum = 0
count = 0

for i, num in enumerate(nums):
    current_sum += num
    complement = current_sum - k
    found = prefix_count.get(complement, 0)
    count += found
    prefix_count[current_sum] = prefix_count.get(current_sum, 0) + 1
    print(f"i={i}, num={num}, sum={current_sum}, need={complement}, found={found}, count={count}, map={prefix_count}")

# i=0: sum=1, need=-2, found=0, count=0
# i=1: sum=3, need=0, found=1 (empty prefix!), count=1  → subarray [1,2]
# i=2: sum=6, need=3, found=1 (prefix sum 3 at i=1), count=2  → subarray [3]

---
### Pattern 7: Sliding Window + HashMap

**The Idea:** Use a hash map to track the frequency of elements inside the current
sliding window. Expand right to add, shrink left to remove.

**When to use:**
- "Longest substring without repeating characters"
- "Minimum window substring"
- "Longest substring with at most K distinct characters"
- Any **substring/subarray** problem where you need to track **unique counts** or **frequencies**

**Template:**
```python
def sliding_window_map(s):
    freq = {}  # char → count in current window
    left = 0
    result = 0
    
    for right in range(len(s)):
        # Expand: add s[right] to window
        freq[s[right]] = freq.get(s[right], 0) + 1
        
        # Shrink: while window is invalid
        while window_is_invalid(freq):
            freq[s[left]] -= 1
            if freq[s[left]] == 0:
                del freq[s[left]]
            left += 1
        
        # Update result
        result = max(result, right - left + 1)
    
    return result
```

In [ ]:
# Pattern 7 Example: Longest Substring Without Repeating Characters
# Find the length of the longest substring with all unique characters.

def length_of_longest_substring(s):
    char_index = {}  # char → last seen index
    left = 0
    result = 0
    
    for right in range(len(s)):
        # If char already in window, jump left pointer past it
        if s[right] in char_index and char_index[s[right]] >= left:
            left = char_index[s[right]] + 1
        
        char_index[s[right]] = right
        result = max(result, right - left + 1)
    
    return result

print(length_of_longest_substring("abcabcbb"))  # 3 ("abc")
print(length_of_longest_substring("bbbbb"))      # 1 ("b")
print(length_of_longest_substring("pwwkew"))     # 3 ("wke")

In [ ]:
# Pattern 7 Example: Minimum Window Substring
# Find the smallest window in s that contains all characters of t.

from collections import Counter

def min_window(s, t):
    if not t or not s:
        return ""
    
    need = Counter(t)        # Characters we need
    window = {}              # Characters in current window
    have = 0                 # How many unique chars we have enough of
    need_count = len(need)   # How many unique chars we need
    
    result = ""
    result_len = float('inf')
    left = 0
    
    for right in range(len(s)):
        # Expand window
        ch = s[right]
        window[ch] = window.get(ch, 0) + 1
        
        # Check if this char is now satisfied
        if ch in need and window[ch] == need[ch]:
            have += 1
        
        # Shrink window while it's valid
        while have == need_count:
            # Update result
            window_len = right - left + 1
            if window_len < result_len:
                result_len = window_len
                result = s[left:right + 1]
            
            # Remove left char
            ch_left = s[left]
            window[ch_left] -= 1
            if ch_left in need and window[ch_left] < need[ch_left]:
                have -= 1
            left += 1
    
    return result

print(min_window("ADOBECODEBANC", "ABC"))  # "BANC"
print(min_window("a", "a"))                  # "a"
print(min_window("a", "aa"))                 # ""

---
## 4. How to SPOT the Pattern in an Interview

This is the most important section. In an interview, you have ~30 minutes.
You need to identify the right approach in the first 3-5 minutes.

### Decision Tree: "Should I use a HashMap/Set?"

Ask yourself these questions **in order**:

```
1. Does the problem involve SEARCHING for something?
   └─ Yes → HashMap/Set is likely useful

2. Is there a BRUTE FORCE with nested loops?
   └─ Yes, inner loop is searching → Replace inner loop with hash lookup

3. Does it mention PAIRS or TWO values?
   └─ Yes → Pattern 1: Complement Lookup

4. Does it mention COUNTING, FREQUENCY, or ANAGRAM?
   └─ Yes → Pattern 2: Frequency Map

5. Does it mention DUPLICATES or UNIQUE?
   └─ Yes → Pattern 3: Seen Set

6. Does it mention GROUPING similar items?
   └─ Yes → Pattern 4: Group By Key

7. Does it need INDICES, not just values?
   └─ Yes → Pattern 5: Index Map

8. Does it involve SUBARRAY SUM?
   └─ Yes → Pattern 6: Prefix Sum + HashMap

9. Does it involve SUBSTRING with constraints?
   └─ Yes → Pattern 7: Sliding Window + HashMap
```

### Keyword Triggers

When you hear these words in a problem, think HashMap/Set:

| Keyword/Phrase | Pattern to Try |
|---------------|----------------|
| "two numbers that sum to" | Complement Lookup |
| "pair", "complement" | Complement Lookup |
| "anagram", "permutation" | Frequency Map |
| "frequency", "count", "most common" | Frequency Map |
| "duplicate", "repeating" | Seen Set |
| "unique", "distinct" | Seen Set |
| "consecutive" | Seen Set (convert to set) |
| "group", "categorize", "bucket" | Group By Key |
| "return indices" | Index Map |
| "subarray sum" | Prefix Sum + HashMap |
| "contiguous subarray" | Prefix Sum + HashMap |
| "substring with at most K" | Sliding Window + HashMap |
| "minimum window" | Sliding Window + HashMap |
| "O(n)" requirement on O(n²) brute force | HashMap optimization |

### The Biggest Signal

**If your brute force is O(n²) with a search in the inner loop, use a HashMap to make it O(n).**

This alone covers 70%+ of HashMap interview problems.

---
## 5. Complexity Analysis

### Time Complexity

| Operation | Average | Worst Case |
|-----------|---------|------------|
| Insert (dict/set) | O(1) | O(n) — hash collision |
| Lookup (dict/set) | O(1) | O(n) — hash collision |
| Delete (dict/set) | O(1) | O(n) — hash collision |
| Build from n items | O(n) | O(n²) — all collide |

**In interviews**, always say O(1) average and mention that worst case is O(n)
if the interviewer asks. They almost never do — O(1) is the expected answer.

### Space Complexity

| Pattern | Extra Space |
|---------|-------------|
| Frequency map | O(n) or O(k) where k = unique elements |
| Seen set | O(n) |
| Prefix sum map | O(n) |
| Character map (26 letters) | O(1) — bounded alphabet |

**Pro tip:** If the input is only lowercase English letters, the space for a
frequency map is O(26) = O(1). Mention this in interviews.

---
## 6. Common Gotchas

### Gotcha 1: Unhashable Types as Keys

Lists, sets, and dicts **cannot** be dictionary keys or set members.
Convert to tuple first.

In [ ]:
# Unhashable type gotcha

# BAD: lists can't be keys
try:
    d = {}
    d[[1, 2]] = "value"  # TypeError!
except TypeError as e:
    print(f"Error: {e}")

# GOOD: convert to tuple
d = {}
d[tuple([1, 2])] = "value"
print(d)  # {(1, 2): 'value'}

# This comes up in:
# - Group anagrams (sorted list → tuple)
# - Graph visited states (list of positions → tuple)
# - Any time you need a list as a key

### Gotcha 2: KeyError vs Default Values

In [ ]:
# KeyError — accessing missing key directly
freq = {}
try:
    freq["a"] += 1  # KeyError! 'a' doesn't exist
except KeyError as e:
    print(f"KeyError: {e}")

# Fix 1: .get() with default
freq["a"] = freq.get("a", 0) + 1
print(freq)  # {'a': 1}

# Fix 2: defaultdict
from collections import defaultdict
freq = defaultdict(int)
freq["a"] += 1  # Works! default is 0
print(dict(freq))  # {'a': 1}

# Fix 3: Counter (for counting)
from collections import Counter
freq = Counter()
freq["a"] += 1
print(freq["z"])  # 0 (no KeyError)

### Gotcha 3: Modifying Dict While Iterating

In [ ]:
# BAD: modifying during iteration
d = {"a": 1, "b": 2, "c": 3}
try:
    for key in d:
        if d[key] < 2:
            del d[key]  # RuntimeError!
except RuntimeError as e:
    print(f"Error: {e}")

# GOOD: iterate over a copy of keys
d = {"a": 1, "b": 2, "c": 3}
for key in list(d.keys()):
    if d[key] < 2:
        del d[key]
print(d)  # {'b': 2, 'c': 3}

# BETTER: build a new dict with comprehension
d = {"a": 1, "b": 2, "c": 3}
d = {k: v for k, v in d.items() if v >= 2}
print(d)  # {'b': 2, 'c': 3}

### Gotcha 4: Counter Comparison Surprises

In [ ]:
from collections import Counter

# Counter ignores zero and negative counts in some operations
c = Counter(a=2, b=-1)
print(+c)  # Counter({'a': 2}) — drops non-positive counts!

# Be careful with subtraction
a = Counter("aab")   # {'a': 2, 'b': 1}
b = Counter("ab")    # {'a': 1, 'b': 1}
print(a - b)          # Counter({'a': 1}) — 'b' disappears (count would be 0)

---
## 7. Interview Execution Framework

When you've identified a HashMap/Set problem, follow this exact sequence:

### Step 1: State the Brute Force (30 seconds)
"The brute force would be to check every pair with nested loops, giving O(n²)."

### Step 2: Identify the Optimization (30 seconds)
"I can use a hash map to store [what I've seen / frequencies / prefix sums]
so that the inner search becomes O(1) lookup."

### Step 3: Explain the Approach (1 minute)
"I'll iterate through the array once. For each element, I'll check if [complement/
match/target] exists in my hash map. If not, I'll store the current element."

### Step 4: Handle Edge Cases (30 seconds)
Before coding, mention:
- Empty input
- Single element
- No solution exists
- Duplicates in input
- Negative numbers (if applicable)

### Step 5: Code It (5-10 minutes)
Write clean code using the appropriate pattern template.

### Step 6: Walk Through an Example (2 minutes)
Trace through a small example, showing the hash map state at each step.

### Step 7: State Complexity (30 seconds)
"Time: O(n) — single pass with O(1) lookups. Space: O(n) for the hash map."

In [ ]:
# === FULL INTERVIEW WALKTHROUGH EXAMPLE ===
# Problem: Two Sum — find indices of two numbers that add to target.

# STEP 1: Brute force
# "Check every pair (i, j) — O(n²) time, O(1) space."

# STEP 2: Optimization insight
# "For each num, I need target - num. Instead of searching for it,
#  I can store previously seen values in a hash map for O(1) lookup."

# STEP 3: Approach
# "Single pass. For each element:
#   - Compute complement = target - num
#   - If complement in map, return both indices
#   - Otherwise, store num → index"

# STEP 4: Edge cases
# - Empty array → return []
# - Single element → can't form pair, return []
# - Duplicate values → works because we check before storing
# - Negative numbers → works, subtraction handles it

# STEP 5: Code
def two_sum(nums, target):
    seen = {}  # value → index
    for i, num in enumerate(nums):
        complement = target - num
        if complement in seen:
            return [seen[complement], i]
        seen[num] = i
    return []

# STEP 6: Trace example
# nums = [2, 7, 11, 15], target = 9
# i=0: num=2, comp=7, seen={} → not found, seen={2:0}
# i=1: num=7, comp=2, seen={2:0} → found! return [0, 1]

# STEP 7: Complexity
# Time: O(n) — one pass, O(1) lookup each
# Space: O(n) — hash map with up to n entries

print(two_sum([2, 7, 11, 15], 9))  # [0, 1]

---
## Bonus: Advanced HashMap Techniques

These come up in harder interviews.

In [ ]:
# Technique: Using tuples as keys for multi-dimensional state
# Example: Count paths in a grid with HashMap memoization

# Track (row, col) states
visited = set()
visited.add((0, 0))
visited.add((1, 2))
print((0, 0) in visited)  # True
print((1, 1) in visited)  # False

# Track complex states
# Example: (position, items_collected, keys_held)
state = (3, frozenset(["coin", "key"]), True)
visited.add(state)

In [ ]:
# Technique: Two-pass HashMap
# Sometimes you need TWO passes: build the map first, then query it.

# Example: Find if any two elements have difference equal to k
def has_pair_with_diff(nums, k):
    num_set = set(nums)  # Pass 1: build set
    for num in nums:      # Pass 2: query
        if num + k in num_set:  # Looking for num + k
            return True
    return False

print(has_pair_with_diff([1, 5, 3, 4, 2], 2))  # True (5-3, 4-2, 3-1)
print(has_pair_with_diff([1, 5, 3, 4, 2], 10)) # False

In [ ]:
# Technique: Encode state as string key
# When the "signature" is complex, convert to a string or tuple

# Example: Check if two words have the same character pattern
# "egg" → "0.1.1"  (same pattern as "add" → "0.1.1")
# "foo" → "0.1.1"  (same pattern)
# "abc" → "0.1.2"  (different pattern)

def pattern_key(word):
    mapping = {}
    pattern = []
    counter = 0
    for ch in word:
        if ch not in mapping:
            mapping[ch] = counter
            counter += 1
        pattern.append(str(mapping[ch]))
    return ".".join(pattern)

print(pattern_key("egg"))   # "0.1.1"
print(pattern_key("add"))   # "0.1.1"  — same!
print(pattern_key("abc"))   # "0.1.2"  — different

---
## Summary: The 7 Patterns at a Glance

| # | Pattern | Data Structure | Key Insight |
|---|---------|---------------|-------------|
| 1 | Complement Lookup | dict or set | Store seen, check complement |
| 2 | Frequency Map | Counter/dict | Count occurrences, compare distributions |
| 3 | Seen Set | set | O(1) membership checks for duplicate/unique |
| 4 | Group By Key | defaultdict(list) | Compute signature → group by it |
| 5 | Index Map | dict (val→idx) | Track positions for distance/index queries |
| 6 | Prefix Sum + Map | dict (sum→count) | Running sum, check if sum-k seen before |
| 7 | Window + Map | dict (char→count) | Track frequencies in sliding window |

**Master these 7 patterns and you can solve 80%+ of HashMap/Set interview problems.**